# Chapter 2 — A Prompt Is Not Yet a Program

**Book alignment:** DSPy From First Principles, Chapter 2

**Question this notebook isolates:** Does swapping the execution strategy (Predict vs ChainOfThought) preserve the caller's contract while changing only how the task is attempted?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy
from common.data import teaching_cases
from common.dspy_program import EditorialRewriteProgram, RewriteSentence


## Same contract, two strategies

Both modules wrap the identical signature class, so calling code passes the same fields either way. `ChainOfThought` adds an internal reasoning step; the task-facing contract does not move. Modules are constructed but never executed against a model.


In [ ]:
direct = dspy.Predict(RewriteSentence)
reasoned = dspy.ChainOfThought(RewriteSentence)

print("Predict inputs: ", sorted(direct.signature.input_fields))
print("CoT inputs:     ", sorted(reasoned.predict.signature.input_fields))
print("Predict outputs:", sorted(direct.signature.output_fields))
print("CoT outputs:    ", sorted(reasoned.predict.signature.output_fields))


In [ ]:
assert set(direct.signature.input_fields) == set(reasoned.predict.signature.input_fields)
assert set(direct.signature.output_fields) == set(RewriteSentence.output_fields)
assert set(RewriteSentence.output_fields) < set(reasoned.predict.signature.output_fields)
assert "reasoning" in reasoned.predict.signature.output_fields
print("contract fixed, strategy swapped, caller unchanged")


## The boundary absorbs failure

A prompt cannot handle its own failure; Chapter 1's brace-finding parser lives in the caller because the string enforces nothing. A module boundary gives failure policy a place to live. The guard below mirrors the `forward` policy in `EditorialRewriteProgram` and is exercised here on deterministic fixtures.


In [ ]:
ed001_sentence = {c.case_id: c for c in teaching_cases()}["ed-001"].sentence


def forward_guard(sentence: str, rewritten_text: str, rationale: str) -> dict:
    if not rewritten_text.strip():
        return {"rewritten_text": sentence, "rationale": "empty output; original retained"}
    return {"rewritten_text": rewritten_text, "rationale": rationale}


empty_case = forward_guard(ed001_sentence, "   ", "whatever the model said")
ok_case = forward_guard(ed001_sentence, "Jalen opened the door, looked inside, afraid.", "tightened")
print("empty  ->", empty_case)
print("normal ->", ok_case)


In [ ]:
assert empty_case["rewritten_text"] == ed001_sentence
assert empty_case["rationale"] == "empty output; original retained"
assert ok_case["rewritten_text"] != ed001_sentence
print("failure policy lives at the boundary, versioned with the program")


## Inspectable structure

A string has no parts to enumerate. The composed program exposes named stages and serializable state, so later chapters can fingerprint, log, and compare candidates instead of diffing prose.


In [ ]:
program = EditorialRewriteProgram()
predictors = dict(program.named_predictors())
state = program.dump_state()
print("predictors:", sorted(predictors))
print("state keys:", sorted(state))


In [ ]:
assert {"analyze", "rewrite", "assess"} <= set(predictors)
assert isinstance(state, dict) and len(state) > 0
print("program parts are enumerable, not inferred from a string")


## What we earned

The behavior now has a boundary: task contract, execution strategy, model configuration, examples, and evaluation can become separately named variables, and failure has a place to live.

Notebook 03 / Chapter 3 turns to the contract at that boundary — what the model should consume and produce, and how to tell a strong contract from a weak one before it costs five chapters.
